# Near-duplicate and shared-source check for the ARCADE stenosis images (W1)

**No training. No GPU needed. About 5 to 10 minutes.**

Question: can frames of the same patient or the same angiography run sit in different folds (or in the pool and the test set)? ARCADE gives no patient id, so this notebook looks for it from the images:
1. exact copies (hash);
2. the thesis rule (32x32 correlation > 0.98) and a stricter one (high-pass 128x128 correlation, plus SSIM) for every pair of the 1500 stenosis images;
3. a positive control: how a slightly shifted / re-contrasted copy of the same image scores, so the thresholds mean something;
4. which fold each similar pair falls in (the folds are rebuilt exactly as in the thesis: StratifiedKFold, 5 splits, seed 42, stratified by min(lesions, 3));
5. whether consecutive file ids are more alike than random pairs (frames of one run would be next to each other);
6. if a syntax folder is found, syntax images against the stenosis pool and test set.

**Input:** ARCADE dataset (nirmalgaud), same as the other notebooks. **Output:** `dup_outputs.zip` - send it back.

In [1]:
import os, json, hashlib, itertools, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")
SEED, STENOSIS_CATEGORY_ID, IMG = 42, 26, 512
ON_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else (Path("/content") if Path("/content").exists() else Path.cwd())
OUT = BASE / "dup_outputs"; OUT.mkdir(parents=True, exist_ok=True)

def find_root():
    for root in [Path(p) for p in os.environ.get("UNI_DATA_ROOTS", "/kaggle/input").split(":")] + [Path.home() / ".cache" / "kagglehub"]:
        if root.exists():
            c = [p for p in root.rglob("train.json") if "stenosis" in str(p).lower()]
            if c: return c[0].parent.parent.parent
    raise RuntimeError("ARCADE dataset not found. Kaggle: Add Input -> 'arcade-dataset' (nirmalgaud).")
ROOT = find_root(); print("ARCADE stenosis root:", ROOT)

def load(split):
    coco = json.load(open(ROOT / split / "annotations" / f"{split}.json"))
    nles = {}
    for a in coco["annotations"]:
        if a["category_id"] == STENOSIS_CATEGORY_ID: nles[a["image_id"]] = nles.get(a["image_id"], 0) + 1
    ims = sorted(coco["images"], key=lambda d: d["id"])
    arr = np.stack([cv2.imread(str(ROOT / split / "images" / im["file_name"]), cv2.IMREAD_GRAYSCALE) for im in ims])
    return arr, [f"{split}/{im['file_name']}" for im in ims], np.array([nles.get(im["id"], 0) for im in ims])
tr, trn, trl = load("train"); va, van, val_ = load("val"); te, ten, tel = load("test")
POOL = np.concatenate([tr, va]); pool_names = trn + van; pool_les = np.concatenate([trl, val_])
N_POOL, N_TEST = len(POOL), len(te)
print("pool", N_POOL, "test", N_TEST, "| shapes", POOL.shape[1:], te.shape[1:])

# same five folds as the thesis (StratifiedKFold, seed 42, stratified by min(lesions, 3))
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(N_POOL), np.minimum(pool_les, 3)))
fold_of = np.zeros(N_POOL, int)
for f, (_, v) in enumerate(FOLDS): fold_of[v] = f
print("fold sizes", [len(v) for _, v in FOLDS])

ALL = np.concatenate([POOL, te]); names = pool_names + ten
split_of = np.array(["pool"] * N_POOL + ["test"] * N_TEST)
fold_all = np.concatenate([fold_of, -np.ones(N_TEST, int)])
N = len(ALL)

# ---- optional: syntax images (used for the vessel head)
SYN = None
cand = [p for p in ROOT.parent.rglob("*") if p.is_dir() and "syntax" in p.name.lower() and (p / "train" / "images").exists()] if ROOT.parent.exists() else []
if cand:
    syn_dir = cand[0]; syn = []
    for sp in ["train", "val", "test"]:
        d = syn_dir / sp / "images"
        if d.exists():
            for f in sorted(d.iterdir()):
                g = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
                if g is not None: syn.append((f"syntax/{sp}/{f.name}", g))
    SYN = syn; print("syntax images found:", len(SYN), "in", syn_dir)
else: print("syntax folder not found: syntax images are not included in this check")

# ---- features
def small(g, s): return cv2.resize(g, (s, s), interpolation=cv2.INTER_AREA).astype(np.float32)
def zvec(a): a = a.reshape(len(a), -1); a = a - a.mean(1, keepdims=True); return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
def hp(a, sig=3):                                   # high-pass: removes slow background so vessel structure decides
    return a - cv2.GaussianBlur(a, (0, 0), sig)
def feats(imgs):
    g32 = np.stack([small(g, 32) for g in imgs]); g128 = np.stack([small(g, 128) for g in imgs])
    return zvec(g32), zvec(np.stack([hp(x) for x in g128])), g128
imgs_all = list(ALL) + ([g for _, g in SYN] if SYN else [])
names_all = names + ([n for n, _ in SYN] if SYN else [])
Z32, ZHP, G128 = feats(imgs_all)
n_all = len(imgs_all)
print("features ready", n_all)

# ---- exact duplicates
md5 = [hashlib.md5(g.tobytes()).hexdigest() for g in imgs_all]
ex = pd.Series(md5).value_counts(); print("exact-duplicate groups:", int((ex > 1).sum()))

# ---- correlation matrices (thesis measure = 32x32 raw correlation, plus a stricter high-pass 128x128 one)
C32 = Z32 @ Z32.T; CHP = ZHP @ ZHP.T
np.fill_diagonal(C32, -1); np.fill_diagonal(CHP, -1)

# ---- positive control: the same image after a small shift, contrast change and noise (what a near-copy would look like)
rng = np.random.default_rng(0); pc = []
for i in rng.choice(N, min(200, N), replace=False):
    g = imgs_all[i].astype(np.float32); dx, dy = rng.integers(-6, 7, 2)
    g2 = np.roll(np.roll(g, dx, 1), dy, 0) * rng.uniform(0.85, 1.15) + rng.normal(0, 4, g.shape)
    z = zvec(np.stack([hp(small(np.clip(g2, 0, 255), 128))]))[0]
    pc.append(float(ZHP[i] @ z))
pc = np.array(pc); print("positive control (shifted/contrast/noise copy), high-pass corr: median %.3f, 5th pct %.3f" % (np.median(pc), np.percentile(pc, 5)))
TH_HP = max(float(np.percentile(pc, 5)), 0.5)
if np.median(pc) < 0.7: print("WARNING: the positive control is weak - high-pass correlation is not a reliable detector on this data")

# ---- nearest neighbour for every image (restricted to the stenosis images, N)
def nn_stats(C, mask_a, mask_b):
    sub = C[np.ix_(mask_a, mask_b)]; return sub.max(1)
isP = np.zeros(n_all, bool); isP[:N_POOL] = True
isT = np.zeros(n_all, bool); isT[N_POOL:N] = True
nn = {}
for nm, C in [("raw32", C32), ("hp128", CHP)]:
    nn[(nm, "pool->pool")] = nn_stats(C, isP, isP); nn[(nm, "test->pool")] = nn_stats(C, isT, isP)
rows = []
for (nm, kind), v in nn.items():
    rows.append(dict(measure=nm, comparison=kind, n=len(v), median=round(float(np.median(v)), 3), p95=round(float(np.percentile(v, 95)), 3), p99=round(float(np.percentile(v, 99)), 3), max=round(float(v.max()), 3)))
NNSUM = pd.DataFrame(rows); NNSUM.to_csv(OUT / "nearest_neighbour_summary.csv", index=False); print(NNSUM.to_string(index=False))

# ---- SSIM on candidate pairs (full 128 px, gaussian window)
def ssim(a, b):
    a, b = a.astype(np.float32), b.astype(np.float32); C1, C2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    mu = lambda x: cv2.GaussianBlur(x, (11, 11), 1.5)
    ma, mb = mu(a), mu(b); saa, sbb, sab = mu(a * a) - ma ** 2, mu(b * b) - mb ** 2, mu(a * b) - ma * mb
    return float((((2 * ma * mb + C1) * (2 * sab + C2)) / ((ma ** 2 + mb ** 2 + C1) * (saa + sbb + C2))).mean())

# ---- candidate pairs: any pair among the stenosis images above a loose high-pass threshold, or raw32 above 0.95
iu = np.triu_indices(N, 1)
vh, v32 = CHP[:N, :N][iu], C32[:N, :N][iu]
sel = np.where((vh >= 0.5) | (v32 >= 0.95))[0]
sel = sel[np.argsort(-vh[sel])][:3000]
pairs = []
for k in sel:
    i, j = int(iu[0][k]), int(iu[1][k])
    kind = "pool-test" if split_of[i] != split_of[j] else ("same fold" if fold_all[i] == fold_all[j] else "different folds")
    pairs.append(dict(img_a=names[i], img_b=names[j], split_a=split_of[i], split_b=split_of[j], fold_a=int(fold_all[i]), fold_b=int(fold_all[j]), kind=kind,
                      corr_raw32=round(float(v32[k]), 4), corr_hp128=round(float(vh[k]), 4), ssim128=round(ssim(G128[i], G128[j]), 4)))
PAIRS = pd.DataFrame(pairs, columns=["img_a","img_b","split_a","split_b","fold_a","fold_b","kind","corr_raw32","corr_hp128","ssim128"]); PAIRS.to_csv(OUT / "candidate_pairs.csv", index=False)
print("candidate pairs kept:", len(PAIRS)); print(PAIRS.head(15).to_string(index=False))

# ---- how many pairs at each threshold, and how they sit relative to the folds
rows = []
for lab, cond in [("raw32 >= 0.98 (thesis rule)", PAIRS.corr_raw32 >= 0.98), ("hp128 >= %.2f (5th pct of positive control)" % TH_HP, PAIRS.corr_hp128 >= TH_HP),
                  ("hp128 >= 0.70", PAIRS.corr_hp128 >= 0.70), ("hp128 >= 0.50", PAIRS.corr_hp128 >= 0.50), ("ssim128 >= 0.80", PAIRS.ssim128 >= 0.80)]:
    s = PAIRS[cond]; rows.append(dict(rule=lab, pairs=len(s), same_fold=int((s.kind == "same fold").sum()), different_folds=int((s.kind == "different folds").sum()), pool_test=int((s.kind == "pool-test").sum())))
TAB = pd.DataFrame(rows); TAB.to_csv(OUT / "pairs_by_threshold.csv", index=False); print(TAB.to_string(index=False))

# ---- clusters (connected components) at the positive-control threshold, among stenosis images
import scipy.sparse as sp, scipy.sparse.csgraph as cg
A = np.zeros((N, N), bool); s = PAIRS[PAIRS.corr_hp128 >= TH_HP]
idx = {n: k for k, n in enumerate(names)}
for _, r in s.iterrows(): A[idx[r.img_a], idx[r.img_b]] = A[idx[r.img_b], idx[r.img_a]] = True
ncomp, lab = cg.connected_components(sp.csr_matrix(A), directed=False)
sizes = np.bincount(lab); multi = [c for c in range(ncomp) if sizes[c] > 1]
cl = []
for c in multi:
    m = np.where(lab == c)[0]; cl.append(dict(cluster=c, size=len(m), n_pool=int((split_of[m] == "pool").sum()), n_test=int((split_of[m] == "test").sum()), folds=sorted(set(int(x) for x in fold_all[m] if x >= 0)), images=";".join(names[k] for k in m[:12])))
CL = pd.DataFrame(cl); CL.to_csv(OUT / "clusters.csv", index=False)
print("clusters with >1 image:", len(CL)); print(CL.head(20).to_string(index=False))

# ---- neighbours in file order: are consecutive ids more alike than random pairs? (frames of one run would be adjacent)
def adj(offset_idx):
    return float(np.mean([CHP[i, j] for i, j in offset_idx]))
cons = [(i, i + 1) for i in range(N_POOL - 1)]
rnd = [tuple(rng.choice(N_POOL, 2, replace=False)) for _ in range(5000)]
print("mean hp corr: consecutive file ids %.3f | random pool pairs %.3f" % (adj(cons), adj(rnd)))

# ---- syntax images vs stenosis test and vs stenosis pool
if SYN:
    S0 = N; syn_idx = np.arange(S0, n_all)
    rows = []
    for nm, C in [("raw32", C32), ("hp128", CHP)]:
        for tgt, mk in [("stenosis test", isT), ("stenosis pool (1200)", isP)]:
            v = C[np.ix_(syn_idx, np.where(mk)[0])].max(1)
            rows.append(dict(measure=nm, syntax_vs=tgt, median=round(float(np.median(v)), 3), p99=round(float(np.percentile(v, 99)), 3), max=round(float(v.max()), 3), n_ge_098_or_tp=int((v >= (0.98 if nm == "raw32" else TH_HP)).sum())))
    SYNT = pd.DataFrame(rows); SYNT.to_csv(OUT / "syntax_vs_stenosis.csv", index=False); print(SYNT.to_string(index=False))

# ---- figure
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for a, nm in zip(ax, ["raw32", "hp128"]):
    a.hist(nn[(nm, "pool->pool")], bins=40, alpha=.6, label="pool -> pool"); a.hist(nn[(nm, "test->pool")], bins=40, alpha=.6, label="test -> pool")
    a.set_title("Nearest-neighbour correlation (%s)" % nm); a.set_xlabel("correlation"); a.legend()
plt.tight_layout(); plt.savefig(OUT / "nn_hist.png", dpi=130)
# ---- zip
z = BASE / "dup_outputs.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT.iterdir(): zf.write(f, f.name)
print("\nDONE. Send back:", z)

ARCADE stenosis root: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/stenosis
pool 1200 test 300 | shapes (512, 512) (512, 512)
fold sizes [240, 240, 240, 240, 240]
syntax images found: 1500 in /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/syntax
features ready 3000
exact-duplicate groups: 132
positive control (shifted/contrast/noise copy), high-pass corr: median 0.609, 5th pct 0.231
measure comparison    n  median   p95   p99   max
  raw32 pool->pool 1200   0.904 0.981 0.999 1.000
  raw32 test->pool  300   0.920 0.958 0.967 0.971
  hp128 pool->pool 1200   0.674 0.883 0.954 1.000
  hp128 test->pool  300   0.789 0.872 0.883 0.895
candidate pairs kept: 3000
        img_a         img_b split_a split_b  fold_a  fold_b            kind  corr_raw32  corr_hp128  ssim128
train/218.png    val/58.png    pool    pool       3       2 different folds      1.0000      1.0000   1.0000
 train/86.png train/916.png    pool    pool       3       3       same fold      1.0000      1.0000